# Import Libraries 

In [17]:
import pandas as pd
import numpy as np
import matplotlib as plt
import sqlite3

# Connect DB
1. make connection using sqlite3
2. make a cursor to execute queries and interact with db

In [18]:
con = sqlite3.connect('Chinook_Sqlite.sqlite')

In [19]:
cur = con.cursor()

# Explore DB

## Explore DB Tables

In [25]:
res = cur.execute("SELECT name FROM sqlite_master where type='table'")
tables = [table[0] for table in res.fetchall()]

In [28]:
print(f"number of tables in db = {len(tables)}")

number of tables in db = 11


In [29]:
for table in tables: 
    print(table, end=", ")

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track, 

In [31]:
table_size = []
for t in tables: 
    cur.execute(f"Select Count(*) from {t} ;")
    table_size.append((t, cur.fetchone()[0]))

In [40]:
for ts in table_size: 
    print(f"Table `{ts[0]}` size: {ts[1]}")

Table `Album` size: 347
Table `Artist` size: 275
Table `Customer` size: 59
Table `Employee` size: 8
Table `Genre` size: 25
Table `Invoice` size: 412
Table `InvoiceLine` size: 2240
Table `MediaType` size: 5
Table `Playlist` size: 18
Table `PlaylistTrack` size: 8715
Table `Track` size: 3503


## check tables schema 

In [53]:
def get_table_schema(cursor, table_name):
    cursor.execute(f'PRAGMA table_info({table_name});')
    columns = cursor.fetchall()
    return columns

In [55]:
for table in tables:
    print(f"\n---------- Table: {table} ---------")
    columns = get_table_schema(cur, table)
    for col in columns: 
        print(f" {col[1]}  ({col[2]})  {'NOT NULL' if col[3] else ''} {'Primary Key' if col[5] else ''} ")  


---------- Table: Album ---------
 AlbumId  (INTEGER)  NOT NULL Primary Key 
 Title  (NVARCHAR(160))  NOT NULL  
 ArtistId  (INTEGER)  NOT NULL  

---------- Table: Artist ---------
 ArtistId  (INTEGER)  NOT NULL Primary Key 
 Name  (NVARCHAR(120))    

---------- Table: Customer ---------
 CustomerId  (INTEGER)  NOT NULL Primary Key 
 FirstName  (NVARCHAR(40))  NOT NULL  
 LastName  (NVARCHAR(20))  NOT NULL  
 Company  (NVARCHAR(80))    
 Address  (NVARCHAR(70))    
 City  (NVARCHAR(40))    
 State  (NVARCHAR(40))    
 Country  (NVARCHAR(40))    
 PostalCode  (NVARCHAR(10))    
 Phone  (NVARCHAR(24))    
 Fax  (NVARCHAR(24))    
 Email  (NVARCHAR(60))  NOT NULL  
 SupportRepId  (INTEGER)    

---------- Table: Employee ---------
 EmployeeId  (INTEGER)  NOT NULL Primary Key 
 LastName  (NVARCHAR(20))  NOT NULL  
 FirstName  (NVARCHAR(20))  NOT NULL  
 Title  (NVARCHAR(30))    
 ReportsTo  (INTEGER)    
 BirthDate  (DATETIME)    
 HireDate  (DATETIME)    
 Address  (NVARCHAR(70))    
 

## get foriegn keys

In [ ]:
cur.execute("PRAGMA foreign_keys = ON;")

In [57]:
def get_foreign_keys(cursor, table_name):
    cursor.execute(f'PRAGMA foreign_key_list({table_name});')
    fks = cursor.fetchall()
    return fks

In [60]:
for table in tables:
    fks = get_foreign_keys(cur, table)
    if fks:
        print(f"\n---------- foreign keys in `{table}`: ---------")
        for fk in fks: 
            print(f"  {fk[3]} refrences {fk[2]}({fk[4]})")


---------- foreign keys in `Album`: ---------
  ArtistId refrences Artist(ArtistId)

---------- foreign keys in `Customer`: ---------
  SupportRepId refrences Employee(EmployeeId)

---------- foreign keys in `Employee`: ---------
  ReportsTo refrences Employee(EmployeeId)

---------- foreign keys in `Invoice`: ---------
  CustomerId refrences Customer(CustomerId)

---------- foreign keys in `InvoiceLine`: ---------
  TrackId refrences Track(TrackId)
  InvoiceId refrences Invoice(InvoiceId)

---------- foreign keys in `PlaylistTrack`: ---------
  TrackId refrences Track(TrackId)
  PlaylistId refrences Playlist(PlaylistId)

---------- foreign keys in `Track`: ---------
  MediaTypeId refrences MediaType(MediaTypeId)
  GenreId refrences Genre(GenreId)
  AlbumId refrences Album(AlbumId)


## Get tables Indecies

In [81]:
def get_index(cursor, table_name):
    cursor.execute(f'PRAGMA index_list({table_name});')
    indecies = cursor.fetchall()
    return indecies

In [82]:
for table in tables:
    indecies = get_index(cur, table)
    if indecies:
        print(f"\n---------- index in `{table}`: ---------")
        for index in indecies:
            print(f"{index}")


---------- index in `Album`: ---------
(0, 'IFK_AlbumArtistId', 0, 'c', 0)

---------- index in `Customer`: ---------
(0, 'IFK_CustomerSupportRepId', 0, 'c', 0)

---------- index in `Employee`: ---------
(0, 'IFK_EmployeeReportsTo', 0, 'c', 0)

---------- index in `Invoice`: ---------
(0, 'IFK_InvoiceCustomerId', 0, 'c', 0)

---------- index in `InvoiceLine`: ---------
(0, 'IFK_InvoiceLineTrackId', 0, 'c', 0)
(1, 'IFK_InvoiceLineInvoiceId', 0, 'c', 0)

---------- index in `PlaylistTrack`: ---------
(0, 'IFK_PlaylistTrackTrackId', 0, 'c', 0)
(1, 'IFK_PlaylistTrackPlaylistId', 0, 'c', 0)
(2, 'sqlite_autoindex_PlaylistTrack_1', 1, 'pk', 0)

---------- index in `Track`: ---------
(0, 'IFK_TrackMediaTypeId', 0, 'c', 0)
(1, 'IFK_TrackGenreId', 0, 'c', 0)
(2, 'IFK_TrackAlbumId', 0, 'c', 0)


# Query  Data

## 10 best-selling tracks

In [13]:
res = cur.execute("""
    SELECT *
    FROM Invoice AS i JOIN InvoiceLine AS il 
    ON i.InvoiceId = il.InvoiceId
    LIMIT 10; 
""")

In [14]:
res.fetchall()

[(1,
  2,
  '2021-01-01 00:00:00',
  'Theodor-Heuss-Straße 34',
  'Stuttgart',
  None,
  'Germany',
  '70174',
  1.98,
  1,
  1,
  2,
  0.99,
  1),
 (1,
  2,
  '2021-01-01 00:00:00',
  'Theodor-Heuss-Straße 34',
  'Stuttgart',
  None,
  'Germany',
  '70174',
  1.98,
  2,
  1,
  4,
  0.99,
  1),
 (2,
  4,
  '2021-01-02 00:00:00',
  'Ullevålsveien 14',
  'Oslo',
  None,
  'Norway',
  '0171',
  3.96,
  3,
  2,
  6,
  0.99,
  1),
 (2,
  4,
  '2021-01-02 00:00:00',
  'Ullevålsveien 14',
  'Oslo',
  None,
  'Norway',
  '0171',
  3.96,
  4,
  2,
  8,
  0.99,
  1),
 (2,
  4,
  '2021-01-02 00:00:00',
  'Ullevålsveien 14',
  'Oslo',
  None,
  'Norway',
  '0171',
  3.96,
  5,
  2,
  10,
  0.99,
  1),
 (2,
  4,
  '2021-01-02 00:00:00',
  'Ullevålsveien 14',
  'Oslo',
  None,
  'Norway',
  '0171',
  3.96,
  6,
  2,
  12,
  0.99,
  1),
 (3,
  8,
  '2021-01-03 00:00:00',
  'Grétrystraat 63',
  'Brussels',
  None,
  'Belgium',
  '1000',
  5.94,
  7,
  3,
  16,
  0.99,
  1),
 (3,
  8,
  '2021-01-03 00: